In [1]:
import pandas as pd

import sys

sys.path.append("../src")
from training import (
    split_features_target,
    standardize_features
)

from evaluation import (
    load_model,
    make_predictions,
    compute_metrics
)

In [2]:
#caricamento dei dati

TRAIN_PATH = "../data/processed/train.csv"
TEST_PATH = "../data/processed/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training set:", train_df.shape)
print("Test set:", test_df.shape)

Training set: (26048, 109)
Test set: (6513, 109)


In [3]:
#Split Features and Target

X_train, X_test, y_train, y_test = split_features_target(
    train_df,
    test_df
)

Splitting features and target...
Features and target successfully separated.



In [4]:
#standardize i dati

X_train_scaled, X_test_scaled, scaler = standardize_features(
    X_train,
    X_test
)

Standardizing features...
Feature standardization completed.



In [5]:
#caricamento dei modelli

LOGISTIC_MODEL_PATH = "../models/logistic_regression.pkl"
RANDOM_FOREST_MODEL_PATH = "../models/random_forest.pkl"

logistic_model = load_model(LOGISTIC_MODEL_PATH)
random_forest_model = load_model(RANDOM_FOREST_MODEL_PATH)

Loading model from ../models/logistic_regression.pkl...
Model loaded successfully.

Loading model from ../models/random_forest.pkl...
Model loaded successfully.



In [6]:
# Make predictions

logistic_predictions = make_predictions(
    logistic_model,
    X_test_scaled
)

rf_predictions = make_predictions(
    random_forest_model,
    X_test
)

Generating predictions...
Predictions generated successfully.

Generating predictions...
Predictions generated successfully.



In [ ]:
X_test.columns.tolist()

In [8]:
# Analyse des performances par groupe

gender_columns = [
    "sex_Female",
    "sex_Male"
]

race_columns = [
    "race_Amer-Indian-Eskimo",
    "race_Asian-Pac-Islander",
    "race_Black",
    "race_Other",
    "race_White"
]

In [9]:
# Analyse selon le genre 

gender_results = []

for group in gender_columns:

    mask = X_test[group] == 1

    metrics = compute_metrics(
        y_test[mask],
        logistic_predictions[mask]
    )

    gender_results.append({
        "Group": group,
        "Samples": mask.sum(),
        **metrics
    })

gender_results = pd.DataFrame(gender_results)

gender_results

Accuracy: 0.9322

Accuracy: 0.8163



,Group,Samples,Accuracy,Precision,Recall,F1-score
0,sex_Female,2153,0.932188,0.767442,0.554622,0.643902
1,sex_Male,4360,0.816284,0.735950,0.620301,0.673195


In [10]:
# Analyse selon la race

race_results = []

for group in race_columns:

    mask = X_test[group] == 1

    metrics = compute_metrics(
        y_test[mask],
        logistic_predictions[mask]
    )

    race_results.append({
        "Group": group,
        "Samples": mask.sum(),
        **metrics
    })

race_results = pd.DataFrame(race_results)

race_results

Accuracy: 0.8955

Accuracy: 0.8214

Accuracy: 0.9136

Accuracy: 0.9318

Accuracy: 0.8486



,Group,Samples,Accuracy,Precision,Recall,F1-score
0,race_Amer-Indian-Eskimo,67,0.895522,0.500000,0.285714,0.363636
1,race_Asian-Pac-Islander,196,0.821429,0.700000,0.549020,0.615385
2,race_Black,579,0.913644,0.706897,0.554054,0.621212
3,race_Other,44,0.931818,0.250000,1.000000,0.400000
4,race_White,5627,0.848587,0.745577,0.616725,0.675057


The model shows different performance across demographic groups. Gender-based differences are mainly observed in Recall, while race-based results should be interpreted carefully because some racial groups contain only a small number of samples. Therefore, fairness metrics such as Disparate Impact (DI) and Equal Opportunity Difference (EOD) are needed to better assess whether these differences indicate the presence of bias.

In [11]:
# ==========================================================
# Calculate Disparate Impact (Gender)
# ==========================================================

protected_group = "sex_Female"
reference_group = "sex_Male"

protected_mask = X_test[protected_group] == 1
reference_mask = X_test[reference_group] == 1

protected_positive_rate = logistic_predictions[protected_mask].mean()
reference_positive_rate = logistic_predictions[reference_mask].mean()

disparate_impact = (
    protected_positive_rate /
    reference_positive_rate
)

print(f"Protected group ({protected_group}) positive prediction rate: "
      f"{protected_positive_rate:.4f}")

print(f"Reference group ({reference_group}) positive prediction rate: "
      f"{reference_positive_rate:.4f}")

print(f"\nDisparate Impact (DI): {disparate_impact:.4f}")

Protected group (sex_Female) positive prediction rate: 0.0799
Reference group (sex_Male) positive prediction rate: 0.2571

Disparate Impact (DI): 0.3107


In [12]:
# ==========================================================
# Calculate Equal Opportunity Difference (Gender)
# ==========================================================

from sklearn.metrics import recall_score

protected_group = "sex_Female"
reference_group = "sex_Male"

protected_mask = X_test[protected_group] == 1
reference_mask = X_test[reference_group] == 1

protected_tpr = recall_score(
    y_test[protected_mask],
    logistic_predictions[protected_mask]
)

reference_tpr = recall_score(
    y_test[reference_mask],
    logistic_predictions[reference_mask]
)

equal_opportunity_difference = (
    protected_tpr - reference_tpr
)

print(f"Protected group ({protected_group}) TPR: {protected_tpr:.4f}")

print(f"Reference group ({reference_group}) TPR: {reference_tpr:.4f}")

print(f"\nEqual Opportunity Difference (EOD): "
      f"{equal_opportunity_difference:.4f}")

Protected group (sex_Female) TPR: 0.5546
Reference group (sex_Male) TPR: 0.6203

Equal Opportunity Difference (EOD): -0.0657
